# Tugas Latihan — Hands-On 1, 4, dan 6

**Anggota Kelompok**

| No | Nama | NIM |
|----|------|-----|
| 1 | Shalvas Andaldo | 2111519002 |
| 2 | Adhitya Naufal | 2411512026 |
| 3 | Diva Hersa | 2411513034 |

> **Catatan:** Nilai skor platform, parameter sensor, dan parameter baterai pada notebook ini adalah **asumsi yang dinyatakan eksplisit** (nilai tipikal dari datasheet/praktik umum). Silakan sesuaikan dengan data pada modul Hands-On masing-masing; seluruh perhitungan otomatis menyesuaikan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.precision", 3)

---
## Tugas 1 — Hands-On 1: Skenario Baru "Gateway Edge-AI untuk Pemrosesan Citra"

**Tujuan:** membuat skenario baru dengan bobot kriteria berbeda, lalu menganalisis apakah rekomendasi platform berubah.

**Metode:** *Weighted Scoring Model* — skor akhir = Σ (bobot kriteria × skor platform pada kriteria itu). Skor 1–5 (5 = terbaik; untuk konsumsi daya dan biaya, 5 = paling hemat/murah).

- **Skenario A (baseline):** node sensor IoT hemat daya (daya & biaya dominan).
- **Skenario B (baru):** gateway edge-AI untuk pemrosesan citra (performa AI & dukungan kamera dominan).

In [ ]:
platforms = ["ESP32-S3", "Raspberry Pi 4", "Jetson Nano", "STM32H7", "Arduino Uno"]
criteria  = ["Performa AI/Komputasi", "Konsumsi Daya", "Biaya",
             "Konektivitas", "Ekosistem/Software", "Dukungan Kamera/Citra"]

# Skor 1-5 (asumsi tipikal)
scores = pd.DataFrame([
    [2, 5, 5, 4, 4, 3],   # ESP32-S3
    [3, 2, 4, 5, 5, 4],   # Raspberry Pi 4
    [5, 1, 2, 4, 4, 5],   # Jetson Nano
    [2, 4, 3, 2, 3, 3],   # STM32H7
    [1, 5, 5, 1, 3, 1],   # Arduino Uno
], index=platforms, columns=criteria)

w_A = np.array([0.10, 0.25, 0.25, 0.20, 0.10, 0.10])   # baseline: node sensor hemat daya
w_B = np.array([0.30, 0.10, 0.10, 0.15, 0.10, 0.25])   # baru: gateway edge-AI citra
assert np.isclose(w_A.sum(), 1) and np.isclose(w_B.sum(), 1)

bobot = pd.DataFrame({"Skenario A (baseline)": w_A, "Skenario B (edge-AI citra)": w_B}, index=criteria)
display(scores)
display(bobot)

In [ ]:
hasil = pd.DataFrame({
    "Skor A": scores.values @ w_A,
    "Skor B": scores.values @ w_B,
}, index=platforms)
hasil["Rank A"] = hasil["Skor A"].rank(ascending=False).astype(int)
hasil["Rank B"] = hasil["Skor B"].rank(ascending=False).astype(int)
display(hasil.sort_values("Skor B", ascending=False))

best_A, best_B = hasil["Skor A"].idxmax(), hasil["Skor B"].idxmax()
print(f"Rekomendasi Skenario A : {best_A}")
print(f"Rekomendasi Skenario B : {best_B}")
print("Rekomendasi berubah?   :", "YA" if best_A != best_B else "TIDAK")

ax = hasil[["Skor A", "Skor B"]].plot(kind="bar", rot=15)
ax.set_ylabel("Skor tertimbang"); ax.set_title("Perbandingan skor platform: Skenario A vs B")
plt.tight_layout(); plt.show()

### Analisis Sensitivitas
Seberapa besar bobot "Performa AI" harus naik sampai rekomendasi berpindah? Bobot AI divariasikan 0–0,6, sisa bobot dibagi proporsional ke kriteria lain (mengikuti proporsi Skenario B).

In [ ]:
idx_ai = 0
wAI = np.linspace(0, 0.6, 121)
rest = w_B.copy(); rest[idx_ai] = 0; rest = rest / rest.sum()

curves = {}
for p in platforms:
    s = scores.loc[p].values
    curves[p] = [wi * s[idx_ai] + (1 - wi) * (rest * s).sum() for wi in wAI]
curves = pd.DataFrame(curves, index=wAI)

ax = curves.plot()
ax.set_xlabel("Bobot Performa AI"); ax.set_ylabel("Skor tertimbang")
ax.set_title("Sensitivitas rekomendasi terhadap bobot Performa AI")
plt.tight_layout(); plt.show()

winner = curves.idxmax(axis=1)
change = winner[winner != winner.shift()].iloc[1:]
print("Titik perpindahan rekomendasi (bobot AI -> pemenang baru):")
print(change.to_string())

### Kesimpulan Tugas 1
- Pada **Skenario A** (daya & biaya dominan), platform berdaya rendah dan murah (**ESP32-S3**) unggul.
- Pada **Skenario B** (gateway edge-AI citra), bobot Performa AI dan Dukungan Kamera meningkat, sehingga **Jetson Nano** menjadi rekomendasi. **Rekomendasi platform berubah.**
- Analisis sensitivitas menunjukkan perpindahan terjadi saat bobot Performa AI melewati ambang tertentu (lihat grafik dan tabel di atas). Artinya keputusan sangat bergantung pada prioritas aplikasi, bukan pada "platform terbaik" secara mutlak.
- Raspberry Pi 4 menjadi kompromi yang baik bila anggaran daya/biaya masih penting.

---
## Tugas 2 — Hands-On 4: Simulasi ADC 16-bit vs 12-bit untuk Sensor Suhu

**Asumsi simulasi**
- Sensor suhu analog tipe LM35: 10 mV/°C, rentang ukur 0–100 °C (0–1,0 V).
- Tegangan referensi ADC (Vref) = 3,3 V.
- Derau sensor/analog ≈ 1 mV rms (≈ 0,1 °C). Akurasi LM35 sendiri ±0,5 °C (tipikal).
- Sinyal suhu: pola harian 28 ± 4 °C selama 24 jam.

In [ ]:
rng = np.random.default_rng(42)

VREF, SENS = 3.3, 0.010            # V, V/°C (LM35)
t = np.linspace(0, 24, 24*60*6)    # 24 jam, sampel tiap 10 detik
T_true = 28 + 4*np.sin(2*np.pi*(t-9)/24)

def adc(v, bits, vref=VREF):
    n = 2**bits
    code = np.clip(np.round(v / vref * (n - 1)), 0, n - 1)
    return code / (n - 1) * vref

def simulate(bits, noise_mV):
    v = T_true*SENS + rng.normal(0, noise_mV/1000, T_true.size)
    return adc(v, bits)/SENS

rows = []
for bits in (12, 16):
    lsb_mV = VREF/(2**bits - 1)*1000
    for label, noise in (("tanpa derau", 0.0), ("derau 1 mV", 1.0)):
        T_meas = simulate(bits, noise)
        err = T_meas - T_true
        rows.append({"ADC": f"{bits}-bit", "Kondisi": label,
                     "LSB (mV)": lsb_mV, "Resolusi (°C/LSB)": lsb_mV/10,
                     "RMSE (°C)": np.sqrt(np.mean(err**2)),
                     "Error maks (°C)": np.abs(err).max()})
tabel = pd.DataFrame(rows)
display(tabel)

In [ ]:
r = tabel.set_index(["ADC", "Kondisi"])
for kond in ("tanpa derau", "derau 1 mV"):
    e12, e16 = r.loc[("12-bit", kond), "RMSE (°C)"], r.loc[("16-bit", kond), "RMSE (°C)"]
    print(f"{kond:12s}: RMSE 12-bit = {e12:.4f} °C | 16-bit = {e16:.4f} °C | perbaikan = {e12/e16:.1f}x")

# Visualisasi zoom 20 menit
sl = slice(0, 120)
np.random.seed(1)
fig, ax = plt.subplots()
ax.plot(t[sl]*60, T_true[sl], "k-", lw=2, label="Suhu sebenarnya")
ax.step(t[sl]*60, adc(T_true[sl]*SENS, 12)/SENS, where="mid", label="ADC 12-bit")
ax.step(t[sl]*60, adc(T_true[sl]*SENS, 16)/SENS, where="mid", label="ADC 16-bit")
ax.set_xlabel("Menit"); ax.set_ylabel("Suhu (°C)"); ax.legend()
ax.set_title("Kuantisasi 12-bit vs 16-bit (tanpa derau, zoom 20 menit)")
plt.tight_layout(); plt.show()

In [ ]:
# Total error budget untuk satu pengukuran (akar jumlah kuadrat)
q12 = (VREF/4095/SENS)/np.sqrt(12)     # std kuantisasi 12-bit (°C)
q16 = (VREF/65535/SENS)/np.sqrt(12)    # std kuantisasi 16-bit (°C)
noise_C = 0.1                          # derau analog (°C rms)
sensor_acc = 0.5                       # akurasi LM35 (°C)

budget = pd.DataFrame({
    "Kuantisasi (°C)": [q12, q16],
    "Derau analog (°C)": [noise_C]*2,
    "Akurasi sensor (°C)": [sensor_acc]*2,
}, index=["12-bit", "16-bit"])
budget["Total RSS (°C)"] = np.sqrt((budget**2).sum(axis=1))
display(budget)
print(f"Peningkatan total error 12-bit -> 16-bit: {(1 - budget.loc['16-bit','Total RSS (°C)']/budget.loc['12-bit','Total RSS (°C)'])*100:.2f}%")

### Kesimpulan Tugas 2
- Resolusi ADC naik **16×** (65.536 vs 4.096 level): LSB turun dari ≈0,81 mV (≈0,08 °C) menjadi ≈0,05 mV (≈0,005 °C).
- Pada kondisi ideal tanpa derau, error kuantisasi memang turun ≈16×.
- **Namun untuk sensor suhu, peningkatannya tidak signifikan secara praktis.** Resolusi 12-bit sudah ≈0,08 °C, jauh lebih halus daripada derau analog (≈0,1 °C) dan terutama akurasi sensor (±0,5 °C). Dominasi *error budget* ada pada sensor dan derau, bukan kuantisasi, sehingga total error nyaris tidak berubah (perbaikan < 1%).
- ADC 16-bit hanya bermanfaat bila rantai sinyal dipertajam (sensor presisi tinggi seperti RTD/termistor terkalibrasi, penguat *low-noise*, Vref stabil). Untuk aplikasi suhu umum, 12-bit cukup, dan 16-bit menambah biaya, waktu konversi, dan konsumsi daya.
- Catatan: dengan Vref 3,3 V dan LM35 yang hanya memakai 0–1 V, sebagian rentang ADC terbuang. Memperkecil Vref atau memakai penguat (gain) lebih efektif meningkatkan presisi daripada hanya menaikkan jumlah bit.

---
## Tugas 3 — Hands-On 6: Estimasi Umur Baterai (Interval Kirim 1 Menit vs 5 Menit)

**Model:** satu siklus = fase aktif (bangun, ukur, kirim data) + fase *deep sleep*.

$$I_{avg} = \frac{I_{aktif}\cdot t_{aktif} + I_{sleep}\cdot (T - t_{aktif})}{T}, \qquad \text{Umur} = \frac{C_{baterai}\cdot \eta}{I_{avg}}$$

**Asumsi (node ESP32 + Wi-Fi/MQTT):** baterai 2000 mAh, efisiensi/derating η = 0,8, arus aktif 120 mA selama 3 detik, arus *deep sleep* 10 µA.

In [ ]:
C_mAh, ETA = 2000, 0.8
I_ACTIVE, T_ACTIVE = 120.0, 3.0     # mA, detik
I_SLEEP = 0.010                     # mA (10 µA)

def battery_life(interval_min):
    T = interval_min*60
    i_avg = (I_ACTIVE*T_ACTIVE + I_SLEEP*(T - T_ACTIVE)) / T
    hours = C_mAh*ETA / i_avg
    return i_avg, hours, hours/24

rows = []
for m in (5, 1):
    i, h, d = battery_life(m)
    rows.append({"Interval (menit)": m, "Pengiriman/hari": 24*60//m,
                 "Arus rata-rata (mA)": i, "Umur (jam)": h, "Umur (hari)": d, "Umur (bulan)": d/30})
hasil3 = pd.DataFrame(rows).set_index("Interval (menit)")
display(hasil3)

d5, d1 = hasil3.loc[5, "Umur (hari)"], hasil3.loc[1, "Umur (hari)"]
print(f"Umur baterai turun dari {d5:.1f} hari menjadi {d1:.1f} hari (≈{d5/d1:.1f}x lebih pendek).")

In [ ]:
intervals = np.array([0.25, 0.5, 1, 2, 5, 10, 15, 30, 60])
days = [battery_life(m)[2] for m in intervals]

fig, ax = plt.subplots()
ax.semilogx(intervals, days, "o-")
for m in (1, 5):
    ax.annotate(f"{m} mnt: {battery_life(m)[2]:.0f} hari", (m, battery_life(m)[2]),
                textcoords="offset points", xytext=(8, 8))
ax.set_xlabel("Interval kirim data (menit, skala log)"); ax.set_ylabel("Umur baterai (hari)")
ax.set_title("Trade-off: interval kirim vs umur baterai")
plt.tight_layout(); plt.show()

# Sensitivitas terhadap durasi aktif (mis. koneksi Wi-Fi lebih lama)
sens = pd.DataFrame({f"t_aktif={ta}s": [(C_mAh*ETA/((I_ACTIVE*ta + I_SLEEP*(m*60-ta))/(m*60)))/24 for m in (5, 1)]
                     for ta in (1, 3, 5)}, index=["5 menit", "1 menit"])
print("Umur baterai (hari) untuk beberapa durasi fase aktif:")
display(sens)

### Trade-off: *Responsivitas Monitoring* vs *Umur Baterai*
- Mengubah interval dari 5 menit menjadi 1 menit membuat frekuensi kirim naik **5×** (288 → 1.440 kali/hari). Karena fase aktif (Wi-Fi/radio) jauh lebih boros (≈120 mA) dibanding *deep sleep* (≈0,01 mA), arus rata-rata naik hampir **5×** (≈1,2 mA → ≈6,0 mA), sehingga umur baterai turun dari ≈55 hari menjadi ≈11 hari.
- **Keuntungan interval 1 menit:** *responsivitas* monitoring lebih tinggi, perubahan kondisi terdeteksi maksimal 1 menit setelah terjadi, data lebih rapat untuk analisis/alarm dini, dan peristiwa singkat lebih kecil kemungkinan terlewat.
- **Kerugiannya:** baterai cepat habis, biaya perawatan (penggantian/pengisian) naik, dan beban jaringan/server meningkat.
- Hubungannya **hampir berbanding terbalik**: umur baterai ∝ interval, selama konsumsi sleep dapat diabaikan. Menggandakan responsivitas ≈ separuh umur baterai.
- **Strategi kompromi:** (1) *adaptive/event-driven reporting* (kirim cepat hanya saat nilai berubah melebihi ambang, selebihnya interval panjang); (2) kurangi durasi fase aktif (koneksi cepat, kirim *batch*, protokol ringan seperti LoRa/BLE/MQTT-SN); (3) kapasitas baterai lebih besar atau panel surya; (4) ukur lokal lebih sering tetapi kirim data lebih jarang.
- Pemilihan interval harus mengikuti kebutuhan aplikasi: monitoring suhu ruangan yang berubah lambat cukup 5–15 menit, sedangkan sistem peringatan kritis membutuhkan interval pendek atau mekanisme *event-driven*.

---
## Ringkasan Akhir
| Tugas | Temuan utama |
|-------|--------------|
| 1. Platform edge-AI citra | Rekomendasi berubah dari ESP32-S3 (hemat daya) ke Jetson Nano (performa AI) saat bobot kriteria diubah. |
| 2. ADC 16-bit vs 12-bit | Presisi kuantisasi naik 16×, tetapi tidak signifikan untuk sensor suhu karena error didominasi derau dan akurasi sensor. |
| 3. Interval kirim 1 menit | Umur baterai turun ≈5× (≈55 → ≈11 hari); responsivitas dibayar dengan umur baterai. |